# Muon Smart Pixel Dataset Analysis Dump

This notebook analyzes the Muon Collider smart-pixel dataset produced by `MuC_Smartpix_Data_Production/launchDataProduction.py`.

It is intentionally sample-first. Metadata scans count every parquet row cheaply, while image and distribution plots use bounded samples so the notebook can run on a multi-million-event dataset without loading the whole dataset into memory.

Dataset format assumptions used here come from this repo's production code:

- `Parquet_Files/*_labels_*.parquet` contains truth/geometry/timing fields.
- `Parquet_Files/*_recon3D_*.parquet` contains 20 time slices of 13 x 21 pixels, flattened to 5460 columns.
- `Parquet_Files/*_recon2D_*.parquet` contains the final time slice, flattened to 273 columns.
- Filename prefixes define the class/source: `signal`, `bib_mm`, `bib_mp`.
- Rows in matching `labels`, `recon2D`, and `recon3D` files are aligned by row index.

Run this notebook from the repository root on AP23. The default path is the dataset you said you generated: `Data_Files/Data_Set_20260612_165954`.


In [ ]:
from __future__ import annotations

import math
import os
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

N_TIME = 20
N_Y = 13
N_X = 21
N_RECON3D = N_TIME * N_Y * N_X
N_RECON2D = N_Y * N_X
KINDS = ("signal", "bib_mm", "bib_mp")

# Main knob. Change this if you move the dataset or run on a different machine.
DATASET_DIR = Path("Data_Files/Data_Set_20260612_165954")
if not DATASET_DIR.exists():
    fallback = Path("/scratch/mbileska/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_20260612_165954")
    if fallback.exists():
        DATASET_DIR = fallback

PARQUET_DIR = DATASET_DIR / "Parquet_Files"
PIXELAV_DIR = DATASET_DIR / "PixelAV"
TRACKLIST_DIR = DATASET_DIR / "Track_Lists"
OUTPUT_DIR = DATASET_DIR / "Analysis_Dump"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sampling knobs. These do not affect exact row-count metadata tables.
MAX_FILES_PER_KIND = 4
MAX_EVENTS_PER_FILE = 2_000
MAX_PIXEL_VALUES = 250_000
RANDOM_STATE = 7

# Event display knobs.
EVENT_KIND = "signal"       # signal, bib_mm, bib_mp
EVENT_FILE_INDEX = 0         # actual file index from filename, not ordinal position
EVENT_ROW_INDEX = 0          # row within that parquet file
EVENT_VIEW = "raw"          # raw, log2, symlog

print("DATASET_DIR:", DATASET_DIR.resolve())
print("PARQUET_DIR exists:", PARQUET_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## Dataset Discovery And Validation

This section checks that every row-aligned parquet triplet exists and that the image widths match this repo's writer:

- `labels`: truth table, variable number of columns
- `recon2D`: 273 columns = `13*21`
- `recon3D`: 5460 columns = `20*13*21`


In [ ]:
_FILE_RE = re.compile(r"^(?P<kind>signal|bib_mm|bib_mp)_(?P<role>labels|recon2D|recon3D)_(?P<index>\d+)\.parquet$")


def parse_parquet_name(path: Path) -> Optional[dict]:
    match = _FILE_RE.match(path.name)
    if match is None:
        return None
    item = match.groupdict()
    item["index"] = int(item["index"])
    item["path"] = path
    return item


def parquet_meta(path: Optional[Path]) -> dict:
    if path is None or not path.exists():
        return {"exists": False, "rows": np.nan, "cols": np.nan, "size_mb": np.nan}
    pf = pq.ParquetFile(path)
    return {
        "exists": True,
        "rows": int(pf.metadata.num_rows),
        "cols": int(pf.metadata.num_columns),
        "size_mb": path.stat().st_size / 1024**2,
    }


def discover_parquet_manifest(parquet_dir: Path = PARQUET_DIR) -> pd.DataFrame:
    parsed = []
    for path in sorted(parquet_dir.glob("*.parquet")):
        item = parse_parquet_name(path)
        if item is not None:
            parsed.append(item)
    if not parsed:
        raise FileNotFoundError(f"No expected parquet files found under {parquet_dir}")

    by_key: Dict[Tuple[str, int], Dict[str, Path]] = {}
    for item in parsed:
        key = (item["kind"], item["index"])
        by_key.setdefault(key, {})[item["role"]] = item["path"]

    rows = []
    for (kind, index), roles in sorted(by_key.items(), key=lambda x: (KINDS.index(x[0][0]), x[0][1])):
        label_path = roles.get("labels")
        recon2d_path = roles.get("recon2D")
        recon3d_path = roles.get("recon3D")
        lm = parquet_meta(label_path)
        r2 = parquet_meta(recon2d_path)
        r3 = parquet_meta(recon3d_path)
        rows.append({
            "kind": kind,
            "file_index": index,
            "label_path": label_path,
            "recon2D_path": recon2d_path,
            "recon3D_path": recon3d_path,
            "label_rows": lm["rows"],
            "recon2D_rows": r2["rows"],
            "recon3D_rows": r3["rows"],
            "label_cols": lm["cols"],
            "recon2D_cols": r2["cols"],
            "recon3D_cols": r3["cols"],
            "label_size_mb": lm["size_mb"],
            "recon2D_size_mb": r2["size_mb"],
            "recon3D_size_mb": r3["size_mb"],
            "has_all_three": bool(lm["exists"] and r2["exists"] and r3["exists"]),
            "rows_match": bool(lm["rows"] == r2["rows"] == r3["rows"]),
            "recon2D_width_ok": bool(r2["cols"] == N_RECON2D),
            "recon3D_width_ok": bool(r3["cols"] == N_RECON3D),
        })
    return pd.DataFrame(rows)


manifest = discover_parquet_manifest()
summary = manifest.groupby("kind", as_index=False).agg(
    files=("file_index", "count"),
    rows=("label_rows", "sum"),
    label_mb=("label_size_mb", "sum"),
    recon2D_mb=("recon2D_size_mb", "sum"),
    recon3D_mb=("recon3D_size_mb", "sum"),
    complete=("has_all_three", "all"),
    row_aligned=("rows_match", "all"),
    recon2D_width_ok=("recon2D_width_ok", "all"),
    recon3D_width_ok=("recon3D_width_ok", "all"),
)
summary.loc[len(summary)] = {
    "kind": "TOTAL",
    "files": int(summary["files"].sum()),
    "rows": int(summary["rows"].sum()),
    "label_mb": float(summary["label_mb"].sum()),
    "recon2D_mb": float(summary["recon2D_mb"].sum()),
    "recon3D_mb": float(summary["recon3D_mb"].sum()),
    "complete": bool(summary["complete"].all()),
    "row_aligned": bool(summary["row_aligned"].all()),
    "recon2D_width_ok": bool(summary["recon2D_width_ok"].all()),
    "recon3D_width_ok": bool(summary["recon3D_width_ok"].all()),
}

display(summary)
display(manifest.head(10))

problems = manifest.query("not has_all_three or not rows_match or not recon2D_width_ok or not recon3D_width_ok")
if len(problems):
    display(problems)
    raise RuntimeError("Dataset manifest has missing or mis-shaped parquet files. Inspect the table above before continuing.")
else:
    print("Manifest validation passed.")


## Label Table Inspection

The label table is produced by `datagensinglefile.makeParquet`. These are the expected columns:

`x-entry`, `y-entry`, `z-entry`, `n_x`, `n_y`, `n_z`, `number_eh_pairs`, `y-local`, `z-global`, `pt`, `hit_time`, `PID`, `cotAlpha`, `cotBeta`, `y-midplane`, `x-midplane`, `adjusted_hit_time`, `adjusted_hit_time_30ps_gaussian`, `adjusted_hit_time_60ps_gaussian`.


In [ ]:
def selected_pairs(manifest: pd.DataFrame, max_files_per_kind: int = MAX_FILES_PER_KIND) -> pd.DataFrame:
    pieces = []
    for kind in KINDS:
        part = manifest[manifest["kind"] == kind].sort_values("file_index").head(max_files_per_kind)
        if len(part):
            pieces.append(part)
    return pd.concat(pieces, ignore_index=True) if pieces else manifest.iloc[0:0].copy()


def read_label_sample(
    manifest: pd.DataFrame,
    max_files_per_kind: int = MAX_FILES_PER_KIND,
    max_events_per_file: int = MAX_EVENTS_PER_FILE,
) -> pd.DataFrame:
    frames = []
    for row in selected_pairs(manifest, max_files_per_kind).itertuples(index=False):
        df = pd.read_parquet(row.label_path, engine="pyarrow")
        df = df.head(max_events_per_file).copy()
        df.insert(0, "kind", row.kind)
        df.insert(1, "file_index", int(row.file_index))
        df.insert(2, "row_index", np.arange(len(df), dtype=int))
        frames.append(df)
    if not frames:
        raise ValueError("No label files selected for sampling.")
    labels = pd.concat(frames, ignore_index=True)
    return add_label_derived_features(labels)


def add_label_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if {"n_x", "n_y", "n_z"}.issubset(out.columns):
        norm = np.sqrt(out["n_x"]**2 + out["n_y"]**2 + out["n_z"]**2)
        out["direction_norm"] = norm
    if "cotAlpha" in out.columns:
        out["alpha_rad"] = np.arctan(out["cotAlpha"].to_numpy(dtype=float))
    if "cotBeta" in out.columns:
        out["beta_rad"] = np.arctan(out["cotBeta"].to_numpy(dtype=float))
    if "pt" in out.columns:
        out["abs_pt"] = np.abs(out["pt"].to_numpy(dtype=float))
    if "z-global" in out.columns:
        # This matches the downstream generator's current approximation.
        out["nModule_raw_guess"] = np.floor(out["z-global"] / 13.0)
        out["x_local_raw_guess"] = np.mod(out["z-global"], 13.0)
    if "number_eh_pairs" in out.columns:
        out["eh_overflow_cap"] = out["number_eh_pairs"] >= 150000
    out["is_signal"] = out["kind"].eq("signal")
    out["is_bib"] = ~out["is_signal"]
    return out


label_sample = read_label_sample(manifest)
print("sampled label rows:", len(label_sample))
print("label columns:")
print(label_sample.columns.tolist())
display(label_sample.head())

display(label_sample.groupby("kind").size().rename("sample_rows").reset_index())
key_cols = [c for c in ["pt", "abs_pt", "cotAlpha", "cotBeta", "alpha_rad", "beta_rad", "y-local", "z-global", "hit_time", "adjusted_hit_time", "number_eh_pairs", "PID"] if c in label_sample.columns]
display(label_sample.groupby("kind")[key_cols].describe().T)


In [ ]:
def plot_hist_by_kind(df: pd.DataFrame, columns: List[str], bins: int = 80, logy: bool = False, clip_quantile: Optional[float] = 0.995):
    n = len(columns)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.6 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, columns):
        values = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if values.empty:
            ax.set_title(f"{col}: no finite values")
            continue
        lo = values.quantile(1.0 - clip_quantile) if clip_quantile else values.min()
        hi = values.quantile(clip_quantile) if clip_quantile else values.max()
        if np.isclose(lo, hi):
            lo, hi = values.min(), values.max()
        for kind in KINDS:
            part = pd.to_numeric(df.loc[df["kind"] == kind, col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            if len(part):
                ax.hist(part, bins=bins, range=(lo, hi), histtype="step", linewidth=1.5, label=kind)
        ax.set_title(col)
        ax.set_ylabel("events")
        if logy:
            ax.set_yscale("log")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    for ax in axes[n:]:
        ax.axis("off")
    return fig

label_plot_cols = [c for c in [
    "abs_pt", "pt", "cotAlpha", "cotBeta", "alpha_rad", "beta_rad",
    "y-local", "z-global", "x-entry", "y-entry", "z-entry",
    "hit_time", "adjusted_hit_time", "number_eh_pairs", "PID",
    "nModule_raw_guess", "x_local_raw_guess", "direction_norm",
] if c in label_sample.columns]

fig = plot_hist_by_kind(label_sample, label_plot_cols, logy=True)
fig.suptitle("Sampled label distributions by source", fontsize=14)
plt.show()


In [ ]:
# PID and overflow summaries are simple but often catch production surprises.
if "PID" in label_sample.columns:
    pid_table = pd.crosstab(label_sample["kind"], label_sample["PID"], margins=True)
    display(pid_table)

if "eh_overflow_cap" in label_sample.columns:
    overflow = label_sample.groupby("kind")["eh_overflow_cap"].agg(["count", "sum", "mean"]).reset_index()
    overflow = overflow.rename(columns={"sum": "events_at_150k_cap", "mean": "fraction_at_150k_cap"})
    display(overflow)


## Recon3D Sample Metrics

This section reads a bounded sample of `recon3D` files and computes event-level metrics without keeping all images in memory.

Important derived metrics:

- `total_charge_all`: sum of all 20 time slices
- `last_frame_charge`: sum of time slice 19
- `active_pixels_all`: number of nonzero pixel-time samples
- `x_size_last`, `y_size_last`: number of active columns/rows in the final frame
- `peak_time_slice`: time slice with the largest total charge
- charge-weighted centroids and widths for final and integrated images


In [ ]:
def _safe_centroid_and_width(image: np.ndarray, axis: int) -> Tuple[np.ndarray, np.ndarray]:
    # image shape: (events, y, x). axis=1 gives y coordinate, axis=2 gives x coordinate.
    if axis == 1:
        profile = image.sum(axis=2)
        coords = np.arange(N_Y, dtype=np.float32)
    elif axis == 2:
        profile = image.sum(axis=1)
        coords = np.arange(N_X, dtype=np.float32)
    else:
        raise ValueError("axis must be 1 or 2")
    denom = profile.sum(axis=1)
    valid = denom > 0
    mean = np.full(profile.shape[0], np.nan, dtype=np.float32)
    width = np.full(profile.shape[0], np.nan, dtype=np.float32)
    if valid.any():
        prof = profile[valid]
        den = denom[valid]
        m = (prof * coords.reshape(1, -1)).sum(axis=1) / den
        var = (prof * (coords.reshape(1, -1) - m.reshape(-1, 1))**2).sum(axis=1) / den
        mean[valid] = m
        width[valid] = np.sqrt(np.maximum(var, 0))
    return mean, width


def _read_recon_rows(path: Path, max_events: int) -> np.ndarray:
    df = pd.read_parquet(path, engine="pyarrow")
    if max_events is not None:
        df = df.head(max_events)
    arr = df.to_numpy(dtype=np.float32, copy=False)
    if arr.shape[1] != N_RECON3D:
        raise ValueError(f"Expected {N_RECON3D} columns in {path}, found {arr.shape[1]}")
    return arr.reshape(arr.shape[0], N_TIME, N_Y, N_X)


def sample_recon3d_metrics(
    manifest: pd.DataFrame,
    max_files_per_kind: int = MAX_FILES_PER_KIND,
    max_events_per_file: int = MAX_EVENTS_PER_FILE,
    max_pixel_values: int = MAX_PIXEL_VALUES,
    random_state: int = RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)
    metric_frames = []
    pixel_values = []
    positive_pixel_values = []
    summaries = {}

    for row in selected_pairs(manifest, max_files_per_kind).itertuples(index=False):
        images = _read_recon_rows(row.recon3D_path, max_events_per_file)
        n = images.shape[0]
        if n == 0:
            continue
        kind = row.kind
        last = images[:, -1, :, :]
        integrated = images.sum(axis=1)
        trace = images.sum(axis=(2, 3))
        total = trace.sum(axis=1)
        last_charge = last.sum(axis=(1, 2))
        active_all = (images != 0).sum(axis=(1, 2, 3))
        active_last = (last != 0).sum(axis=(1, 2))
        x_profile_last = last.sum(axis=1)
        y_profile_last = last.sum(axis=2)
        x_size_last = (x_profile_last != 0).sum(axis=1)
        y_size_last = (y_profile_last != 0).sum(axis=1)
        peak_time = trace.argmax(axis=1)
        max_pixel = images.max(axis=(1, 2, 3))
        last_y_centroid, last_y_width = _safe_centroid_and_width(last, axis=1)
        last_x_centroid, last_x_width = _safe_centroid_and_width(last, axis=2)
        int_y_centroid, int_y_width = _safe_centroid_and_width(integrated, axis=1)
        int_x_centroid, int_x_width = _safe_centroid_and_width(integrated, axis=2)

        metrics = pd.DataFrame({
            "kind": kind,
            "file_index": int(row.file_index),
            "row_index": np.arange(n, dtype=int),
            "total_charge_all": total,
            "last_frame_charge": last_charge,
            "active_pixels_all": active_all,
            "active_pixels_last": active_last,
            "active_fraction_all": active_all / float(N_RECON3D),
            "active_fraction_last": active_last / float(N_RECON2D),
            "x_size_last": x_size_last,
            "y_size_last": y_size_last,
            "peak_time_slice": peak_time,
            "max_pixel_charge": max_pixel,
            "last_x_centroid": last_x_centroid,
            "last_y_centroid": last_y_centroid,
            "last_x_width": last_x_width,
            "last_y_width": last_y_width,
            "integrated_x_centroid": int_x_centroid,
            "integrated_y_centroid": int_y_centroid,
            "integrated_x_width": int_x_width,
            "integrated_y_width": int_y_width,
        })
        metric_frames.append(metrics)

        flat = images.reshape(-1)
        if flat.size:
            take = min(max_pixel_values // max(1, len(selected_pairs(manifest, max_files_per_kind))), flat.size)
            idx = rng.choice(flat.size, size=take, replace=False)
            pixel_values.append(flat[idx])
            positive = flat[flat > 0]
            if positive.size:
                take_pos = min(take, positive.size)
                idx_pos = rng.choice(positive.size, size=take_pos, replace=False)
                positive_pixel_values.append(positive[idx_pos])

        s = summaries.setdefault(kind, {
            "n_events": 0,
            "sum_trace": np.zeros(N_TIME, dtype=np.float64),
            "sum_trace2": np.zeros(N_TIME, dtype=np.float64),
            "nonzero_by_time": np.zeros(N_TIME, dtype=np.float64),
            "sum_integrated_image": np.zeros((N_Y, N_X), dtype=np.float64),
            "sum_last_image": np.zeros((N_Y, N_X), dtype=np.float64),
        })
        s["n_events"] += n
        s["sum_trace"] += trace.sum(axis=0)
        s["sum_trace2"] += (trace**2).sum(axis=0)
        s["nonzero_by_time"] += (images != 0).mean(axis=(2, 3)).sum(axis=0)
        s["sum_integrated_image"] += integrated.sum(axis=0)
        s["sum_last_image"] += last.sum(axis=0)

    metrics_df = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
    pixel_values = np.concatenate(pixel_values) if pixel_values else np.array([], dtype=np.float32)
    positive_pixel_values = np.concatenate(positive_pixel_values) if positive_pixel_values else np.array([], dtype=np.float32)
    return metrics_df, summaries, pixel_values, positive_pixel_values


recon_metrics, recon_summaries, pixel_values, positive_pixel_values = sample_recon3d_metrics(manifest)
print("sampled recon events:", len(recon_metrics))
display(recon_metrics.groupby("kind").size().rename("sampled_events").reset_index())
display(recon_metrics.groupby("kind").describe().T)


In [ ]:
metric_cols = [
    "total_charge_all", "last_frame_charge", "active_pixels_all", "active_pixels_last",
    "active_fraction_all", "active_fraction_last", "x_size_last", "y_size_last",
    "peak_time_slice", "max_pixel_charge", "integrated_x_width", "integrated_y_width",
]
fig = plot_hist_by_kind(recon_metrics, metric_cols, bins=80, logy=True, clip_quantile=0.995)
fig.suptitle("Sampled recon3D event metrics by source", fontsize=14)
plt.show()


In [ ]:
def plot_charge_trends(summaries: dict):
    available = [k for k in KINDS if k in summaries and summaries[k]["n_events"] > 0]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)
    time = np.arange(N_TIME)
    for kind in available:
        s = summaries[kind]
        n = s["n_events"]
        mean_trace = s["sum_trace"] / n
        var_trace = np.maximum(s["sum_trace2"] / n - mean_trace**2, 0)
        std_trace = np.sqrt(var_trace)
        mean_nonzero = s["nonzero_by_time"] / n
        axes[0].plot(time, mean_trace, marker="o", label=kind)
        axes[0].fill_between(time, mean_trace - std_trace, mean_trace + std_trace, alpha=0.12)
        axes[1].plot(time, mean_nonzero, marker="o", label=kind)
        axes[2].plot(time, mean_trace / np.maximum(mean_trace.sum(), 1e-12), marker="o", label=kind)
    axes[0].set_title("Mean charge trace")
    axes[0].set_xlabel("time slice")
    axes[0].set_ylabel("mean charge")
    axes[1].set_title("Mean active pixel fraction")
    axes[1].set_xlabel("time slice")
    axes[1].set_ylabel("fraction of 13x21 pixels active")
    axes[2].set_title("Normalized mean charge trace")
    axes[2].set_xlabel("time slice")
    axes[2].set_ylabel("fraction of event charge")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    return fig

fig = plot_charge_trends(recon_summaries)
plt.show()


In [ ]:
def plot_mean_images(summaries: dict, image_key: str = "sum_integrated_image", title: str = "mean integrated image"):
    available = [k for k in KINDS if k in summaries and summaries[k]["n_events"] > 0]
    values = [summaries[k][image_key] / summaries[k]["n_events"] for k in available]
    vmin = min(float(np.nanmin(v)) for v in values)
    vmax = max(float(np.nanmax(v)) for v in values)
    fig, axes = plt.subplots(1, len(available), figsize=(5.2 * len(available), 4.4), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, kind, image in zip(axes, available, values):
        im = ax.imshow(image, origin="lower", aspect="auto", cmap="magma", vmin=vmin, vmax=vmax)
        ax.set_title(f"{kind}: {title}")
        ax.set_xlabel("x pixel")
        ax.set_ylabel("y pixel")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return fig

fig = plot_mean_images(recon_summaries, "sum_integrated_image", "mean integrated image")
plt.show()

fig = plot_mean_images(recon_summaries, "sum_last_image", "mean last-frame image")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)
if pixel_values.size:
    axes[0].hist(pixel_values, bins=120, histtype="step", log=True)
    axes[0].set_title("Sampled raw pixel-time charges")
    axes[0].set_xlabel("charge")
    axes[0].set_ylabel("sample count")
if positive_pixel_values.size:
    axes[1].hist(positive_pixel_values, bins=120, histtype="step", log=True)
    axes[1].set_title("Positive pixel-time charges")
    axes[1].set_xlabel("charge")
    axes[1].set_ylabel("sample count")
    log2_values = np.log1p(positive_pixel_values) / np.log(2)
    axes[2].hist(log2_values, bins=120, histtype="step", log=True)
    axes[2].set_title("Positive charges after log2(1+x)")
    axes[2].set_xlabel("log2(1+charge)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()


## Label/Recon Correlations

This joins sampled label rows to sampled recon metrics by `(kind, file_index, row_index)`, then shows relationships between truth/geometry/timing and charge-shape metrics.


In [ ]:
joined_sample = recon_metrics.merge(
    label_sample,
    on=["kind", "file_index", "row_index"],
    how="left",
    suffixes=("", "_label"),
)
print("joined rows:", len(joined_sample))

corr_cols = [c for c in [
    "abs_pt", "pt", "cotAlpha", "cotBeta", "alpha_rad", "beta_rad", "y-local", "z-global",
    "hit_time", "adjusted_hit_time", "number_eh_pairs",
    "total_charge_all", "last_frame_charge", "active_pixels_all", "active_pixels_last",
    "x_size_last", "y_size_last", "peak_time_slice", "max_pixel_charge",
    "integrated_x_width", "integrated_y_width",
] if c in joined_sample.columns]

corr = joined_sample[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(0.55 * len(corr_cols) + 4, 0.55 * len(corr_cols) + 3), constrained_layout=True)
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(np.arange(len(corr_cols)), labels=corr_cols, rotation=90)
ax.set_yticks(np.arange(len(corr_cols)), labels=corr_cols)
ax.set_title("Sample correlation matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
def scatter_by_kind(df: pd.DataFrame, x: str, y: str, max_points: int = 25_000, alpha: float = 0.25):
    fig, ax = plt.subplots(figsize=(6.2, 4.8), constrained_layout=True)
    for kind in KINDS:
        part = df[df["kind"] == kind]
        if part.empty or x not in part or y not in part:
            continue
        if len(part) > max_points:
            part = part.sample(max_points, random_state=RANDOM_STATE)
        ax.scatter(part[x], part[y], s=6, alpha=alpha, label=kind)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.grid(alpha=0.3)
    ax.legend()
    return fig

for x, y in [
    ("abs_pt", "total_charge_all"),
    ("cotAlpha", "x_size_last"),
    ("cotBeta", "y_size_last"),
    ("y-local", "integrated_y_centroid"),
    ("z-global", "integrated_x_centroid"),
    ("hit_time", "peak_time_slice"),
]:
    if x in joined_sample.columns and y in joined_sample.columns:
        fig = scatter_by_kind(joined_sample, x, y)
        plt.show()


## Single Event Inspection

Choose a `kind`, `file_index`, and row index. The loader reads the matching label row and `recon3D` image, then the plotter shows:

- integrated 20-slice charge image
- final time-slice image
- x/y profiles vs time
- charge trace
- final-frame profiles


In [ ]:
def resolve_pair(manifest: pd.DataFrame, kind: str, file_index: int) -> pd.Series:
    part = manifest[(manifest["kind"] == kind) & (manifest["file_index"] == int(file_index))]
    if part.empty:
        available = manifest[manifest["kind"] == kind]["file_index"].tolist()[:20]
        raise KeyError(f"No {kind} file_index={file_index}. First available indices: {available}")
    return part.iloc[0]


def load_event(kind: str = EVENT_KIND, file_index: int = EVENT_FILE_INDEX, row_index: int = EVENT_ROW_INDEX) -> dict:
    pair = resolve_pair(manifest, kind, file_index)
    labels = pd.read_parquet(pair.label_path, engine="pyarrow")
    recon = pd.read_parquet(pair.recon3D_path, engine="pyarrow")
    if row_index < 0 or row_index >= len(labels):
        raise IndexError(f"row_index={row_index} outside file with {len(labels)} rows")
    label = labels.iloc[int(row_index)].copy()
    flat = recon.iloc[int(row_index)].to_numpy(dtype=np.float32, copy=False)
    image = flat.reshape(N_TIME, N_Y, N_X)
    return {
        "kind": kind,
        "file_index": int(file_index),
        "row_index": int(row_index),
        "label": label,
        "image": image,
        "label_path": pair.label_path,
        "recon3D_path": pair.recon3D_path,
    }


def transform_image(image: np.ndarray, view: str = "raw") -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    if view == "raw":
        return image
    if view == "log2":
        out = np.zeros_like(image, dtype=np.float32)
        nz = image != 0
        out[nz] = np.sign(image[nz]) * np.log1p(np.abs(image[nz])) / np.log(2)
        return out
    if view == "symlog":
        return np.sign(image) * np.log1p(np.abs(image))
    raise ValueError("view must be raw, log2, or symlog")


def event_metrics(event: dict) -> pd.DataFrame:
    image = event["image"]
    last = image[-1]
    integrated = image.sum(axis=0)
    trace = image.sum(axis=(1, 2))
    row = {
        "kind": event["kind"],
        "file_index": event["file_index"],
        "row_index": event["row_index"],
        "total_charge_all": float(trace.sum()),
        "last_frame_charge": float(last.sum()),
        "active_pixels_all": int((image != 0).sum()),
        "active_pixels_last": int((last != 0).sum()),
        "peak_time_slice": int(trace.argmax()),
        "max_pixel_charge": float(image.max()),
        "x_size_last": int((last.sum(axis=0) != 0).sum()),
        "y_size_last": int((last.sum(axis=1) != 0).sum()),
    }
    return pd.DataFrame([row])


def display_event_label(event: dict):
    label = event["label"].to_frame("value")
    label.index.name = "label"
    display(label)


event = load_event(EVENT_KIND, EVENT_FILE_INDEX, EVENT_ROW_INDEX)
print("Loaded", event["kind"], "file", event["file_index"], "row", event["row_index"])
print("label:", event["label_path"])
print("recon3D:", event["recon3D_path"])
display(event_metrics(event))
display_event_label(event)


In [ ]:
def plot_event_overview(event: dict, view: str = EVENT_VIEW):
    raw = event["image"]
    image = transform_image(raw, view)
    integrated = image.sum(axis=0)
    last = image[-1]
    trace = image.sum(axis=(1, 2))
    y_time = image.sum(axis=2).T
    x_time = image.sum(axis=1).T
    y_last = last.sum(axis=1)
    x_last = last.sum(axis=0)

    heat_values = [integrated, last, y_time, x_time]
    vmin = min(float(np.nanmin(v)) for v in heat_values)
    vmax = max(float(np.nanmax(v)) for v in heat_values)
    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0

    fig = plt.figure(figsize=(16, 11), constrained_layout=True)
    gs = fig.add_gridspec(3, 3)
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[0, 2])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    ax5 = fig.add_subplot(gs[1, 2])
    ax6 = fig.add_subplot(gs[2, :])

    for ax, arr, title, xlabel, ylabel in [
        (ax0, integrated, "integrated over time", "x pixel", "y pixel"),
        (ax1, last, "last time slice", "x pixel", "y pixel"),
        (ax2, raw.sum(axis=0), "raw integrated", "x pixel", "y pixel"),
        (ax3, y_time, "y profile vs time", "time slice", "y pixel"),
        (ax4, x_time, "x profile vs time", "time slice", "x pixel"),
    ]:
        im = ax.imshow(arr, origin="lower", aspect="auto", cmap="magma", vmin=vmin if title != "raw integrated" else None, vmax=vmax if title != "raw integrated" else None)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax5.plot(np.arange(N_Y), y_last, marker="o", label="y profile")
    ax5_t = ax5.twinx()
    ax5_t.plot(np.arange(N_X), x_last, marker="s", color="tab:orange", label="x profile")
    ax5.set_title("last-frame profiles")
    ax5.set_xlabel("pixel index")
    ax5.set_ylabel("y profile charge")
    ax5_t.set_ylabel("x profile charge")
    ax5.grid(alpha=0.3)

    ax6.plot(np.arange(N_TIME), trace, marker="o")
    ax6.set_title("charge trace")
    ax6.set_xlabel("time slice")
    ax6.set_ylabel(f"charge ({view})")
    ax6.grid(alpha=0.3)

    fig.suptitle(f"{event['kind']} file={event['file_index']} row={event['row_index']} view={view}", fontsize=14)
    return fig

fig = plot_event_overview(event, view=EVENT_VIEW)
plt.show()


## Event Movie

The animation uses a fixed color scale across all time slices. Set `EVENT_VIEW` to `raw`, `log2`, or `symlog` above, then rerun the event cells.


In [ ]:
def build_event_animation(event: dict, view: str = EVENT_VIEW, interval: int = 250):
    image = transform_image(event["image"], view)
    vmin = float(np.nanmin(image))
    vmax = float(np.nanmax(image))
    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0
    fig, ax = plt.subplots(figsize=(6.2, 4.8), constrained_layout=True)
    im = ax.imshow(image[0], origin="lower", aspect="auto", cmap="magma", vmin=vmin, vmax=vmax)
    title = ax.set_title("")
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(f"charge ({view})")

    def update(frame):
        im.set_data(image[frame])
        title.set_text(f"{event['kind']} file={event['file_index']} row={event['row_index']} time slice {frame}")
        return im, title

    anim = animation.FuncAnimation(fig, update, frames=N_TIME, interval=interval, blit=False)
    return anim

anim = build_event_animation(event, view=EVENT_VIEW, interval=250)
plt.close(anim._fig)
HTML(anim.to_jshtml())


In [ ]:
# Optional: save the selected event movie. MP4 requires ffmpeg; GIF requires pillow.
SAVE_EVENT_MOVIE = False
MOVIE_FORMAT = "gif"  # gif or mp4

if SAVE_EVENT_MOVIE:
    anim = build_event_animation(event, view=EVENT_VIEW, interval=250)
    if MOVIE_FORMAT == "gif":
        out = OUTPUT_DIR / f"event_{event['kind']}_{event['file_index']}_{event['row_index']}_{EVENT_VIEW}.gif"
        anim.save(out, writer="pillow", fps=4)
    elif MOVIE_FORMAT == "mp4":
        out = OUTPUT_DIR / f"event_{event['kind']}_{event['file_index']}_{event['row_index']}_{EVENT_VIEW}.mp4"
        anim.save(out, writer="ffmpeg", fps=4, dpi=150)
    else:
        raise ValueError("MOVIE_FORMAT must be gif or mp4")
    plt.close(anim._fig)
    print("saved", out)


## Find Interesting Events In The Sample

This searches the sampled metrics table for events that are useful to inspect: largest charge, most active pixels, widest clusters, unusual timing, etc. Copy one row's `(kind, file_index, row_index)` into the event knobs above.


In [ ]:
def top_events(df: pd.DataFrame, metric: str, n: int = 10, ascending: bool = False) -> pd.DataFrame:
    cols = ["kind", "file_index", "row_index", metric, "total_charge_all", "last_frame_charge", "active_pixels_all", "x_size_last", "y_size_last", "peak_time_slice"]
    cols = [c for c in cols if c in df.columns]
    return df.sort_values(metric, ascending=ascending).head(n)[cols]

for metric, ascending in [
    ("total_charge_all", False),
    ("active_pixels_all", False),
    ("x_size_last", False),
    ("y_size_last", False),
    ("peak_time_slice", False),
    ("last_frame_charge", True),
]:
    if metric in recon_metrics:
        print("\nTop events by", metric)
        display(top_events(recon_metrics, metric, n=8, ascending=ascending))


## ML Feature Reconstruction Sanity Check

This mirrors the feature engineering in `OptimizedDataGenerator4_data_shuffled_bigData_NewFormat.py` for the sampled rows:

- raw `recon3D` charges are transformed with `sign(x)*log2(1+abs(x))`
- `x_profile` and `y_profile` are sums over the transformed image, then log-transformed again by the generator
- `x_size`, `y_size`, `nPix`, `y_local`, `z_global`, `nModule`, `x_local`, `inVectAsic` are derived as in the generator

This cell is mainly for checking that feature ranges look sane before training.


In [ ]:
def build_ml_feature_sample(
    manifest: pd.DataFrame,
    max_files_per_kind: int = min(MAX_FILES_PER_KIND, 2),
    max_events_per_file: int = min(MAX_EVENTS_PER_FILE, 1000),
) -> pd.DataFrame:
    frames = []
    for row in selected_pairs(manifest, max_files_per_kind).itertuples(index=False):
        labels = pd.read_parquet(row.label_path, engine="pyarrow").head(max_events_per_file).copy()
        images = _read_recon_rows(row.recon3D_path, max_events_per_file)
        transformed = np.zeros_like(images, dtype=np.float32)
        nz = images != 0
        transformed[nz] = np.sign(images[nz]) * np.log1p(np.abs(images[nz])) / np.log(2)
        last = transformed[:, -1]
        y_profiles = last.sum(axis=2)
        x_profiles = last.sum(axis=1)
        n_pix = (last != 0).sum(axis=(1, 2))
        x_size = (x_profiles != 0).sum(axis=1) / 21.0
        y_size = (y_profiles != 0).sum(axis=1) / 13.0
        # Generator applies another log transform to profiles.
        y_profiles_log = np.where(y_profiles != 0, np.sign(y_profiles) * np.log1p(np.abs(y_profiles)) / np.log(2), 0)
        x_profiles_log = np.where(x_profiles != 0, np.sign(x_profiles) * np.log1p(np.abs(x_profiles)) / np.log(2), 0)
        z_global = labels["z-global"].to_numpy(dtype=np.float32)
        y_local = labels["y-local"].to_numpy(dtype=np.float32)
        part = pd.DataFrame({
            "kind": row.kind,
            "file_index": int(row.file_index),
            "row_index": np.arange(len(labels), dtype=int),
            "ml_x_size": x_size,
            "ml_y_size": y_size,
            "ml_nPix": n_pix,
            "ml_y_local_norm": y_local / 8.5,
            "ml_z_global_norm": z_global / 65.0,
            "ml_total_charge_norm": labels["number_eh_pairs"].to_numpy(dtype=np.float32) / 150000.0,
            "ml_nModule_norm": np.floor(z_global / 13.0) / 5.0,
            "ml_x_local_norm": np.mod(z_global, 13.0) / 13.0,
            "x_profile_log_sum": x_profiles_log.sum(axis=1),
            "y_profile_log_sum": y_profiles_log.sum(axis=1),
        })
        frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

ml_features = build_ml_feature_sample(manifest)
display(ml_features.groupby("kind").describe().T)
fig = plot_hist_by_kind(ml_features, [c for c in ml_features.columns if c.startswith("ml_") or c.endswith("_sum")], bins=80, logy=True)
fig.suptitle("Sampled ML feature ranges", fontsize=14)
plt.show()


## Optional PixelAV Text Output Audit

The parquet files are the analysis source of truth. This optional audit inspects a bounded number of raw PixelAV `.out` files and logs to count `<cluster>` records and overflow messages. It can be slow if you scan every output file, so keep the defaults small unless you are running a batch audit.


In [ ]:
def audit_pixelav_text(pixelav_dir: Path = PIXELAV_DIR, max_files: int = 12) -> pd.DataFrame:
    rows = []
    for path in sorted(pixelav_dir.glob("*_pixelav_*.out"))[:max_files]:
        kind = None
        for k in KINDS:
            if path.name.startswith(k + "_"):
                kind = k
                break
        clusters = 0
        with path.open("r", errors="replace") as handle:
            for line in handle:
                if "<cluster>" in line:
                    clusters += 1
        log_path = path.with_name(path.name.replace("_pixelav_", "_pixelav_log_").replace(".out", ".txt"))
        overflow_lines = 0
        if log_path.exists():
            with log_path.open("r", errors="replace") as handle:
                overflow_lines = sum(1 for line in handle if line.strip())
        rows.append({
            "kind": kind,
            "pixelav_file": path.name,
            "clusters_in_out": clusters,
            "log_file": log_path.name if log_path.exists() else None,
            "nonempty_log_lines": overflow_lines,
            "size_mb": path.stat().st_size / 1024**2,
        })
    return pd.DataFrame(rows)

pixelav_audit = audit_pixelav_text(max_files=12)
display(pixelav_audit)


## Save Sample Tables

This writes lightweight CSV summaries under `Analysis_Dump/` for later comparison between generated datasets.


In [ ]:
SAVE_SAMPLE_TABLES = True
if SAVE_SAMPLE_TABLES:
    manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)
    summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
    label_sample.to_csv(OUTPUT_DIR / "label_sample.csv", index=False)
    recon_metrics.to_csv(OUTPUT_DIR / "recon_metrics_sample.csv", index=False)
    joined_sample.to_csv(OUTPUT_DIR / "joined_sample.csv", index=False)
    ml_features.to_csv(OUTPUT_DIR / "ml_feature_sample.csv", index=False)
    print("wrote CSV summaries to", OUTPUT_DIR)
